In [ ]:
from datasets import load_dataset, Dataset

In [ ]:
ds = load_dataset("Helsinki-NLP/opus-100", "en-fr")

In [ ]:
ds["train"][0]

ds["train"].shape

In [ ]:
MAX_LEN = 50

filtered_train = ds["train"].filter(
    lambda x: len(x["translation"]["fr"].split()) <= MAX_LEN and 
              len(x["translation"]["en"].split()) <= MAX_LEN
)

print(filtered_train.shape)

def length_ratio(example, min_ratio=0.5, max_ratio=2.0):
    fr_len = len(example["translation"]["fr"].split())
    en_len = len(example["translation"]["en"].split())
    if en_len == 0 or fr_len == 0:
        return False
    ratio = fr_len / en_len
    return min_ratio <= ratio <= max_ratio

filtered_train.filter(length_ratio)

print(filtered_train.shape)

def remove_duplicates(dataset):
    seen = set()
    unique_data = []

    for example in dataset:
        pair = (
            example["translation"]["fr"].strip(),
            example["translation"]["en"].strip()
        )
        if pair not in seen:
            seen.add(pair)
            unique_data.append(example)

    
    return Dataset.from_list(unique_data)

cleaned = remove_duplicates(filtered_train)

import re

def clean_chars(ex):
    fr = ex["translation"]["fr"]
    en = ex["translation"]["en"]

    def is_clean(text):
        if len(text.split()) < 3:
            return False
        # Check if the text contains only allowed characters
        allowed_chars = r"^[\w\s.,;!?\"'()\-\–«»À-ÿàâçéèêëîïôûùüÿñæœ…]+$"
        return re.fullmatch(allowed_chars, text) is not None

    return is_clean(fr) and is_clean(en)


cleaned = cleaned.filter(clean_chars)

print(cleaned.shape)

In [ ]:
with open("train.fr", "w", encoding="utf-8") as fr_file, \
     open("train.en", "w", encoding="utf-8") as en_file:
    
    for example in cleaned:
        fr_sentence = example["translation"]["fr"].strip()
        en_sentence = example["translation"]["en"].strip()
        
        fr_file.write(fr_sentence + "\n")
        en_file.write(en_sentence + "\n")